# Debugra Benchmark — Results Analysis

Comparing Debugra (autonomous multi-agent QA) vs. human testers across 18 seeded bugs in two SUTs.

**Protocol**: 5 human testers × 2 SUTs, 30 min each. Debugra run 5× per SUT with fixed seed.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from scipy.stats import mannwhitneyu

plt.rcParams['figure.dpi'] = 130
plt.rcParams['font.family'] = 'sans-serif'

## 1. Load Data

*TODO: replace placeholder data with actual benchmark results.*

In [ ]:
# ── Bug catalog ──────────────────────────────────────────────────────────────
import yaml, pathlib
catalog = yaml.safe_load((pathlib.Path('..') / 'bugs.yaml').read_text())['bugs']
bugs_df = pd.DataFrame(catalog)
bugs_df.head()

In [ ]:
# ── Human tester results (placeholder — replace with form export CSV) ──────
human_raw = {
    'tester':    ['T1','T1','T1','T2','T2','T3','T3','T3','T3','T4','T4','T5','T5','T5'],
    'bug_id':    ['LMS-01','LMS-04','SHOP-04','LMS-01','LMS-02','LMS-04','SHOP-01','SHOP-04','SHOP-07',
                  'LMS-01','SHOP-07','LMS-03','LMS-05','SHOP-04'],
    'time_min':  [3.2, 8.1, 12.4, 4.5, 11.2, 2.1, 6.8, 14.2, 21.0, 5.0, 18.3, 3.9, 9.7, 17.5],
}
human_df = pd.DataFrame(human_raw)
print(f"Human findings: {len(human_df)} across {human_df.tester.nunique()} testers")

In [ ]:
# ── Debugra results (placeholder — replace with exported findings JSONs) ──
debugra_raw = {
    'run':     [1,1,1,1,1,1,1,1,1,1,1,1,1,1,
                2,2,2,2,2,2,2,2,2,2,2,2,2,2],
    'bug_id':  ['LMS-01','LMS-02','LMS-03','LMS-04','LMS-05','LMS-07','LMS-08',
                'SHOP-01','SHOP-02','SHOP-03','SHOP-04','SHOP-05','SHOP-07','SHOP-08',
                'LMS-01','LMS-02','LMS-03','LMS-04','LMS-05','LMS-07','LMS-08',
                'SHOP-01','SHOP-02','SHOP-03','SHOP-04','SHOP-05','SHOP-07','SHOP-08'],
    'oracle':  ['http_5xx','dom_assertion','dom_assertion','console_error','http_5xx',
                'dom_assertion','dom_assertion','dom_assertion','dom_assertion',
                'dom_assertion','dom_assertion','dom_assertion','network_fail','dom_assertion'] * 2,
    'time_s':  list(np.random.randint(30, 240, 28)),
}
debugra_df = pd.DataFrame(debugra_raw)
print(f"Debugra findings: {debugra_df.bug_id.nunique()} unique bugs across {debugra_df.run.nunique()} runs")

## 2. Recall Comparison

In [ ]:
total_bugs = len(bugs_df)

# Per-tester recall
human_recall = human_df.groupby('tester')['bug_id'].nunique() / total_bugs * 100

# Per-run recall
debugra_recall = debugra_df.groupby('run')['bug_id'].nunique() / total_bugs * 100

print(f"Human mean recall:   {human_recall.mean():.1f}% ± {human_recall.std():.1f}%")
print(f"Debugra mean recall: {debugra_recall.mean():.1f}% ± {debugra_recall.std():.1f}%")

stat, pval = mannwhitneyu(debugra_df.groupby('run')['bug_id'].nunique(),
                           human_df.groupby('tester')['bug_id'].nunique(),
                           alternative='greater')
print(f"Mann-Whitney U={stat:.0f}, p={pval:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ── Recall bar chart ─────────────────────────────────────────────────────────
ax = axes[0]
means = [human_recall.mean(), debugra_recall.mean()]
stds  = [human_recall.std(),  debugra_recall.std()]
bars = ax.bar(['Human\n(n=5, 30 min)', 'Debugra\n(n=5 runs)'], means,
               yerr=stds, capsize=5, color=['#94a3b8','#6366f1'], width=0.5)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_ylim(0, 100)
ax.set_title('Bug Recall')
ax.set_ylabel('% of 18 seeded bugs found')
for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, m + 2, f'{m:.0f}%', ha='center', fontsize=11, fontweight='bold')

# ── Severity breakdown ────────────────────────────────────────────────────────
ax2 = axes[1]
sev_order = ['critical','high','medium','low']
colors = {'critical':'#dc2626','high':'#ea580c','medium':'#ca8a04','low':'#2563eb'}

def recall_by_sev(found_ids, catalog_df):
    rows = []
    for sev in sev_order:
        total = len(catalog_df[catalog_df.severity == sev])
        found = sum(1 for b in found_ids if catalog_df[catalog_df.id==b].iloc[0]['severity']==sev) if total else 0
        rows.append({'severity': sev, 'recall': found/total*100 if total else 0})
    return pd.DataFrame(rows)

human_found   = set(human_df.bug_id)
debugra_found = set(debugra_df.bug_id)

x = np.arange(len(sev_order))
w = 0.35
h_rec = recall_by_sev(human_found, bugs_df)
d_rec = recall_by_sev(debugra_found, bugs_df)
ax2.bar(x - w/2, h_rec.recall, w, label='Human', color='#94a3b8')
ax2.bar(x + w/2, d_rec.recall, w, label='Debugra', color='#6366f1')
ax2.set_xticks(x); ax2.set_xticklabels(sev_order, fontsize=11)
ax2.yaxis.set_major_formatter(mtick.PercentFormatter())
ax2.set_ylim(0, 120); ax2.set_title('Recall by Severity')
ax2.legend()

plt.tight_layout()
plt.savefig('figures/recall_comparison.png', bbox_inches='tight')
plt.show()

## 3. Time to First Bug

In [ ]:
human_ttfb  = human_df.groupby('tester')['time_min'].min()
debugra_ttfb = debugra_df.groupby('run')['time_s'].min() / 60

print(f"Human TTFB mean:   {human_ttfb.mean():.1f} min")
print(f"Debugra TTFB mean: {debugra_ttfb.mean():.1f} min")

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.boxplot([human_ttfb.values, debugra_ttfb.values],
           labels=['Human', 'Debugra'], patch_artist=True,
           boxprops=dict(facecolor='#e0e7ff'), medianprops=dict(color='#6366f1', linewidth=2))
ax.set_ylabel('Minutes to first bug')
ax.set_title('Time to First Bug')
plt.tight_layout()
plt.savefig('figures/time_to_first_bug.png', bbox_inches='tight')
plt.show()

## 4. Oracle Coverage

Which detection oracles fired for which bugs?

In [ ]:
oracle_counts = debugra_df.groupby('oracle')['bug_id'].nunique().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(7, 3))
ax.barh(oracle_counts.index, oracle_counts.values, color='#6366f1')
ax.set_xlabel('Unique bugs detected')
ax.set_title('Bugs Detected by Oracle Type')
plt.tight_layout()
plt.savefig('figures/oracle_coverage.png', bbox_inches='tight')
plt.show()

## 5. Summary Table

In [ ]:
summary = pd.DataFrame({
    'Metric': ['Total bugs found (of 18)', 'Critical/High recall', 'Mean time to first bug',
               'Mean run / session duration', 'False positives (det. oracle)'],
    'Debugra': [f"{int(debugra_recall.mean()/100*18)}/18",
                f"{d_rec[d_rec.severity.isin(['critical','high'])].recall.mean():.0f}%",
                f"{debugra_ttfb.mean():.1f} min",
                "~4–5 min",
                "0"],
    'Human (mean)': [f"{human_recall.mean()/100*18:.1f}/18",
                     f"{h_rec[h_rec.severity.isin(['critical','high'])].recall.mean():.0f}%",
                     f"{human_ttfb.mean():.1f} min",
                     "30 min",
                     "—"],
})
summary.set_index('Metric')